# 01 · Data Check & Control Figures

**Goal:** verify the Maven Toys data before building the Power BI model, and calculate **control figures**
(revenue, profit, units by year / category / location). After the dashboard is built, every number in Power BI
must match these. A mismatch means a broken relationship or a wrong measure.

**Source:** [Mexico Toy Sales, Maven Analytics Data Playground](https://mavenanalytics.io/data-playground?search=Mexico%20Toy%20Sales).
Maven Toys is a **fictitious** chain of 50 toy stores in Mexico; the data is for learning purposes.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.float_format", "{:,.2f}".format)
RAW = Path("../data/raw")

sales = pd.read_csv(RAW / "sales.csv", parse_dates=["Date"])
products = pd.read_csv(RAW / "products.csv")
stores = pd.read_csv(RAW / "stores.csv", parse_dates=["Store_Open_Date"])
inventory = pd.read_csv(RAW / "inventory.csv")
calendar = pd.read_csv(RAW / "calendar.csv", encoding="utf-8-sig")

pd.DataFrame({
    "rows": {"sales": len(sales), "products": len(products), "stores": len(stores),
             "inventory": len(inventory), "calendar": len(calendar)},
})

,rows
sales,829262
products,35
stores,50
inventory,1593
calendar,638


## 1. Structure and types

In [2]:
products.head()

,Product_ID,Product_Name,Product_Category,Product_Cost,Product_Price
0,1,Action Figure,Toys,$9.99,$15.99
1,2,Animal Figures,Toys,$9.99,$12.99
2,3,Barrel O' Slime,Art & Crafts,$1.99,$3.99
3,4,Chutes & Ladders,Games,$9.99,$12.99
4,5,Classic Dominoes,Games,$7.99,$9.99


In [3]:
products.dtypes

Product_ID          int64
Product_Name          str
Product_Category      str
Product_Cost          str
Product_Price         str
dtype: object

**Issue:** cost and price are stored as text with a `$` sign and a trailing space (`"$9.99 "`).
In Power BI they must be cleaned in Power Query (remove `$`, trim, change type to *Fixed decimal number*).
Here we do the same in pandas.

In [4]:
for col in ["Product_Cost", "Product_Price"]:
    products[col] = products[col].str.replace(r"[$,\s]", "", regex=True).astype(float)

products["Unit_Margin_%"] = 100 * (products["Product_Price"] - products["Product_Cost"]) / products["Product_Price"]
products.sort_values("Unit_Margin_%").head()

,Product_ID,Product_Name,Product_Category,Product_Cost,Product_Price,Unit_Margin_%
8,9,Dino Egg,Toys,9.99,10.99,9.10
29,30,Rubik's Cube,Games,17.99,19.99,10.01
30,31,Splash Balls,Sports & Outdoors,7.99,8.99,11.12
17,18,Lego Bricks,Toys,34.99,39.99,12.50
18,19,Magic Sand,Art & Crafts,13.99,15.99,12.51


In [5]:
print("Calendar first / last value:", calendar["Date"].iloc[0], "/", calendar["Date"].iloc[-1])
print("Sales period:", sales["Date"].min().date(), "–", sales["Date"].max().date())

Calendar first / last value: 1/1/2022 / 9/30/2023
Sales period: 2022-01-01 – 2023-09-30


- The calendar uses the US format `M/D/YYYY` and the file starts with a BOM. Instead of importing it,
  we will **create the calendar in DAX** (`CALENDAR` + date attributes), which is the recommended approach.
- **Note:** in this version of the dataset sales run from **Jan 2022 to Sep 2023**, so 2023 is a partial year
  (9 months). Year-over-year comparisons must compare the same months.

## 2. Data quality

In [6]:
checks = {
    "sales: duplicate Sale_ID": sales["Sale_ID"].duplicated().sum(),
    "sales: missing values": sales.isna().sum().sum(),
    "sales: units <= 0": (sales["Units"] <= 0).sum(),
    "sales → products: orphans": (~sales["Product_ID"].isin(products["Product_ID"])).sum(),
    "sales → stores: orphans": (~sales["Store_ID"].isin(stores["Store_ID"])).sum(),
    "inventory: duplicate store+product": inventory.duplicated(["Store_ID", "Product_ID"]).sum(),
    "inventory: negative stock": (inventory["Stock_On_Hand"] < 0).sum(),
    "products: price < cost": (products["Product_Price"] < products["Product_Cost"]).sum(),
}
pd.Series(checks, name="count").to_frame()

,count
sales: duplicate Sale_ID,0
sales: missing values,0
sales: units <= 0,0
sales → products: orphans,0
sales → stores: orphans,0
inventory: duplicate store+product,0
inventory: negative stock,0
products: price < cost,0


All checks pass: keys are unique and every foreign key matches, so the model can use simple one-to-many
relationships. `inventory` has no single key: its grain is **store + product** (1,593 rows).

In [7]:
print(stores["Store_Location"].value_counts(), "\n")
print(products["Product_Category"].value_counts(), "\n")
print("Cities:", stores["Store_City"].nunique())

Store_Location
Downtown       29
Commercial     12
Residential     6
Airport         3
Name: count, dtype: int64 

Product_Category
Toys                 9
Art & Crafts         8
Games                8
Sports & Outdoors    7
Electronics          3
Name: count, dtype: int64 

Cities: 29


## 3. Control figures

These numbers are the reference for checking the Power BI measures.

In [8]:
df = sales.merge(products, on="Product_ID").merge(stores, on="Store_ID")
df["Revenue"] = df["Units"] * df["Product_Price"]
df["Cost"] = df["Units"] * df["Product_Cost"]
df["Profit"] = df["Revenue"] - df["Cost"]
df["Year"] = df["Date"].dt.year


def summary(frame):
    out = frame[["Revenue", "Cost", "Profit", "Units"]].sum()
    out["Margin %"] = 100 * out["Profit"] / out["Revenue"]
    out["Transactions"] = len(frame)
    return out


total = summary(df).to_frame("Total")
total

,Total
Revenue,"14,444,572.35"
Cost,"10,430,543.35"
Profit,"4,014,029.00"
Units,"1,090,565.00"
Margin %,27.79
Transactions,"829,262.00"


In [9]:
by_year = df.groupby("Year").apply(summary)
by_year

,Revenue,Cost,Profit,Units,Margin %,Transactions
Year,,,,,,
2022,"7,482,498.08","5,292,711.08","2,189,787.00","549,492.00",29.27,"420,845.00"
2023,"6,962,074.27","5,137,832.27","1,824,242.00","541,073.00",26.20,"408,417.00"


In [10]:
same_months = df[df["Date"].dt.month <= 9].groupby("Year")["Revenue"].sum()
print(f"Revenue Jan–Sep 2022: {same_months[2022]:,.2f}")
print(f"Revenue Jan–Sep 2023: {same_months[2023]:,.2f}")
print(f"YoY growth (same months): {100 * (same_months[2023] / same_months[2022] - 1):.1f}%")

Revenue Jan–Sep 2022: 5,320,115.85
Revenue Jan–Sep 2023: 6,962,074.27
YoY growth (same months): 30.9%


In [11]:
by_category = df.groupby("Product_Category").apply(summary).sort_values("Profit", ascending=False)
by_category

,Revenue,Cost,Profit,Units,Margin %,Transactions
Product_Category,,,,,,
Toys,"5,093,241.00","4,013,714.00","1,079,527.00","267,200.00",21.20,"221,227.00"
Electronics,"2,246,771.25","1,245,334.25","1,001,437.00","134,075.00",44.57,"99,025.00"
Art & Crafts,"2,705,364.26","1,952,010.26","753,354.00","325,574.00",27.85,"220,673.00"
Games,"2,226,836.27","1,552,843.27","673,993.00","194,673.00",30.27,"157,006.00"
Sports & Outdoors,"2,172,359.57","1,666,641.57","505,718.00","169,043.00",23.28,"131,331.00"


In [12]:
by_location = df.groupby("Store_Location").apply(summary).sort_values("Profit", ascending=False)
by_location["Stores"] = stores["Store_Location"].value_counts()
by_location["Profit per store"] = by_location["Profit"] / by_location["Stores"]
by_location

,Revenue,Cost,Profit,Units,Margin %,Transactions,Stores,Profit per store
Store_Location,,,,,,,,
Downtown,"8,219,596.27","5,970,868.27","2,248,728.00","628,073.00",27.36,"480,224.00",29,"77,542.34"
Commercial,"3,279,139.52","2,352,275.52","926,864.00","240,648.00",28.27,"184,705.00",12,"77,238.67"
Residential,"1,656,113.98","1,195,725.98","460,388.00","125,402.00",27.80,"95,876.00",6,"76,731.33"
Airport,"1,289,722.58","911,673.58","378,049.00","96,442.00",29.31,"68,457.00",3,"126,016.33"


In [13]:
by_product = df.groupby("Product_Name").apply(summary).sort_values("Profit", ascending=False)
by_product.head(10)

,Revenue,Cost,Profit,Units,Margin %,Transactions
Product_Name,,,,,,
Colorbuds,"1,564,476.32","729,532.32","834,944.00","104,368.00",53.37,"72,988.00"
Action Figure,"926,748.42","579,000.42","347,748.00","57,958.00",37.52,"48,497.00"
Lego Bricks,"2,388,882.63","2,090,197.63","298,685.00","59,737.00",12.50,"48,030.00"
Deck Of Cards,"587,397.66","335,295.66","252,102.00","84,034.00",42.92,"68,083.00"
Glass Marbles,"412,322.82","224,732.82","187,590.00","37,518.00",45.50,"24,507.00"
Barrel O' Slime,"365,735.37","182,409.37","183,326.00","91,663.00",50.13,"54,078.00"
Kids Makeup Kit,"488,415.67","341,817.67","146,598.00","24,433.00",30.02,"21,648.00"
Nerf Gun,"530,594.57","397,879.57","132,715.00","26,543.00",25.01,"23,709.00"
Dart Gun,"505,092.12","378,740.12","126,352.00","31,588.00",25.02,"26,203.00"


In [14]:
inv = inventory.merge(products, on="Product_ID")
print(f"Stock units:            {inv['Stock_On_Hand'].sum():,}")
print(f"Stock value at cost:    {(inv['Stock_On_Hand'] * inv['Product_Cost']).sum():,.2f}")
print(f"Stock value at retail:  {(inv['Stock_On_Hand'] * inv['Product_Price']).sum():,.2f}")
print(f"Out-of-stock store+product pairs: {(inv['Stock_On_Hand'] == 0).sum()}")

Stock units:            29,742
Stock value at cost:    300,209.58
Stock value at retail:  410,240.58
Out-of-stock store+product pairs: 77


## Summary

| Check | Result |
| --- | --- |
| Data quality | no missing values, duplicates or orphan keys |
| Prices | stored as text (`"$9.99 "`), clean in Power Query |
| Calendar | build in DAX instead of importing the CSV |
| Period | Jan 2022 – Sep 2023 (2023 is partial: compare the same months) |

**First insights from the control figures**
- Revenue Jan–Sep grew **~31% year over year**.
- **Electronics** has the highest margin (~45%) while **Toys** brings the most revenue at the lowest margin (~21%).
- **Lego Bricks** is the #1 product by revenue but only #3 by profit (12.5% unit margin); **Colorbuds** alone
  brings ~21% of total profit.
- **Downtown** stores generate 56% of profit, but only because there are 29 of them. Per store, **Airport**
  locations earn ~126k profit vs. ~77k for every other location type: **1.6× more**.

Next step → build the data model in Power BI (`docs/build_guide.md`).